# MediSense-Cardio -- copy the TRUE lead II of the 140 real-capture recordings (Kaggle, CPU)

**Why:** some clean pages give a wrong heart rate. To find out if the reference or the digitiser is at fault we need the true lead II signal, which is only inside the competition data.
**What it does:** reads the true lead II of the 140 recordings used in Arm C (40) and its extension (100) and saves them in two small files. Nothing else. No GPU, about 2-5 minutes.

**Steps (Kaggle):**
1. New notebook -> File -> Import notebook -> upload this file.
2. Right panel -> Input -> Add Input -> search "PhysioNet - Digitization of ECG Images" -> add it (same as before).
3. Settings: Accelerator = None (CPU is enough). Internet is not needed.
4. Click Save Version -> Save & Run All (Commit). When it finishes, open Output and download `truth_lead_II.npz` and `truth_meta.csv`.
5. Put both files in your Downloads folder and tell me.


In [ ]:
import os, glob, json, datetime
import numpy as np, pandas as pd

IDS = ["10140238", "1059602762", "1067975047", "1135737846", "1154813853", "1177153463", "1189828070", "1278116713", "1312364803", "1380849732", "1381006600", "1386861857", "1395556771", "1411350094", "1419593015", "1445349505", "1475354244", "1484831977", "1502655579", "1503423095", "1524001991", "1542217251", "1557988993", "1683970273", "1722454310", "1783318297", "181086745", "1907398397", "19585145", "1987857200", "2006667321", "202594271", "2042532735", "2101175952", "2140273702", "214041807", "2160317410", "2236932015", "2261781339", "2280120893", "2322831046", "236587707", "2371149418", "2422967768", "2446033601", "2479159356", "2496700414", "2504502933", "2512496238", "2567450244", "2587667307", "2591785253", "2596413640", "261884655", "2634228760", "264050025", "2656943578", "2700749937", "2725753137", "2738115934", "2757222712", "2779122924", "2800037729", "2806926781", "2971605936", "2991598184", "3002700944", "3041779288", "3051591599", "3088924592", "3110635653", "3121342163", "3137394949", "3174963235", "3182381560", "3199374928", "3203822582", "3206606655", "3238781612", "3240145691", "3246341876", "3267417662", "332019944", "3337580882", "334628058", "3356121866", "3378387616", "3542724905", "3543481279", "355772854", "3633367021", "36494663", "3676566153", "3728731089", "3732607929", "3754781958", "3765046489", "3823160274", "3863689498", "3871909495", "3903047348", "3917626689", "3944858755", "3960796436", "4012861469", "4049828212", "4050029009", "405307743", "4072460642", "4092807213", "4146512364", "4178620257", "4204675127", "4215966477", "4225377454", "4229829773", "4241274306", "4246336200", "4246414264", "433501783", "468577389", "54505697", "546219911", "57127905", "57655084", "604355197", "606637005", "622992199", "702673935", "703370585", "704591922", "735091708", "756565655", "791634457", "812510412", "847329431", "867270557", "88853795", "947867923", "950071314"]
COMP_ROOT = '/kaggle/input'      # where Kaggle puts attached data
WORK = '/kaggle/working'         # what is saved and can be downloaded
os.makedirs(WORK, exist_ok=True)
assert len(IDS) == 140 and len(set(IDS)) == 140

found = glob.glob(COMP_ROOT + '/**/train.csv', recursive=True)
assert found, ('The competition data is not attached. Right panel -> Input -> Add Input -> search '
               '"PhysioNet - Digitization of ECG Images" -> add it. Then Run All again.')
COMP_DIR = os.path.dirname(found[0])
train = pd.read_csv(f'{COMP_DIR}/train.csv')
assert 'id' in train.columns and 'fs' in train.columns, ('train.csv does not have the columns id and fs: ' + str(list(train.columns)))
train['id'] = train['id'].astype(str)
train_fs = dict(zip(train['id'], train['fs']))
print('competition folder:', COMP_DIR, '| recordings in train.csv:', len(train))
missing = [i for i in IDS if i not in train_fs or not os.path.exists(f'{COMP_DIR}/train/{i}/{i}.csv')]
print('requested recordings without a true-signal file:', len(missing), missing[:10])
assert not missing, 'some requested recordings are missing: paste this output into the chat'


In [ ]:
arrays, meta = {}, []
for rid in IDS:
    t = pd.read_csv(f'{COMP_DIR}/train/{rid}/{rid}.csv')
    assert 'II' in t.columns, f'no column II in {rid}: {list(t.columns)}'
    ii = t['II'].to_numpy(dtype='float32')              # keep NaN positions, so nothing is shifted
    arrays['r' + rid] = ii
    fin = np.isfinite(ii)
    meta.append({'id': rid, 'fs': float(train_fs[rid]), 'n_rows': int(len(t)), 'n_II': int(fin.sum()), 'n_II_nan': int((~fin).sum()),
                 'duration_s_from_II': round(float(fin.sum()) / float(train_fs[rid]), 3),
                 'II_min_mV': round(float(np.nanmin(ii)), 4), 'II_max_mV': round(float(np.nanmax(ii)), 4),
                 'columns': ' '.join(t.columns)})
np.savez_compressed(f'{WORK}/truth_lead_II.npz', **arrays)
pd.DataFrame(meta).to_csv(f'{WORK}/truth_meta.csv', index=False)
print('saved', len(arrays), 'signals')


In [ ]:
# check what was saved
z = np.load(f'{WORK}/truth_lead_II.npz')
m = pd.read_csv(f'{WORK}/truth_meta.csv')
assert len(z.files) == 140 and len(m) == 140, (len(z.files), len(m))
assert all(len(z['r' + str(i)]) == n for i, n in zip(m['id'], m['n_rows']))
print(m['fs'].value_counts().sort_index().to_string())
print('duration of the true lead II (s): min', m['duration_s_from_II'].min(), '| max', m['duration_s_from_II'].max())
print('recordings with NaN inside lead II:', int((m['n_II_nan'] > 0).sum()))
print('TRUTH EXPORT COMPLETE. Download: truth_lead_II.npz and truth_meta.csv')
